In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score
from sklearn.preprocessing import LabelEncoder

# 1. Load the datasets
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')
sample_sub = pd.read_csv('sample_submission.csv')

print("Data loaded successfully!")
print(f"Train shape: {train.shape}, Test shape: {test.shape}")

Data loaded successfully!
Train shape: (77299, 11), Test shape: (41778, 10)


In [11]:
# Data Cleaning


# To ensure we encode categories consistently, we temporarily combine train and test
train['is_train'] = 1
test['is_train'] = 0
test['demand'] = 0 # Placeholder for test set

full_data = pd.concat([train, test], ignore_index=True)

# 1. Fill Missing Values
# Fill Temperature with the median (average-ish) value
full_data['Temperature'] = full_data['Temperature'].fillna(full_data['Temperature'].median())

# Fill text columns with the most frequent value (the mode)
full_data['RoadType'] = full_data['RoadType'].fillna(full_data['RoadType'].mode()[0])
full_data['Weather'] = full_data['Weather'].fillna(full_data['Weather'].mode()[0])

# 2. Extract Time Features
# The timestamp is like "14:30". We split this into 'hour' (14) and 'minute' (30)
full_data['hour'] = full_data['timestamp'].apply(lambda x: int(x.split(':')[0]))
full_data['minute'] = full_data['timestamp'].apply(lambda x: int(x.split(':')[1]))
full_data = full_data.drop('timestamp', axis=1) # Drop the original text column

# 3. Encode Text to Numbers
# LabelEncoder turns categories into numbers (e.g., Sunny=0, Rainy=1)
text_columns = ['geohash', 'RoadType', 'LargeVehicles', 'Landmarks', 'Weather']
le = LabelEncoder()

for col in text_columns:
    full_data[col] = le.fit_transform(full_data[col].astype(str))

# 4. Split back into Train and Test
train_clean = full_data[full_data['is_train'] == 1].drop('is_train', axis=1)
test_clean = full_data[full_data['is_train'] == 0].drop(['is_train', 'demand'], axis=1)


print("Data cleaning and encoding complete!")


Data cleaning and encoding complete!


In [12]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder

# 1. Load the original datasets
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

# 2. Combine temporarily for consistent cleaning
train['is_train'] = 1
test['is_train'] = 0
test['demand'] = 0 # Placeholder for test set

full_data = pd.concat([train, test], ignore_index=True)

# 3. Fill Missing Values
full_data['Temperature'] = full_data['Temperature'].fillna(full_data['Temperature'].median())
full_data['RoadType'] = full_data['RoadType'].fillna(full_data['RoadType'].mode()[0])
full_data['Weather'] = full_data['Weather'].fillna(full_data['Weather'].mode()[0])

# 4. Extract Time Features
full_data['hour'] = full_data['timestamp'].apply(lambda x: int(x.split(':')[0]))
full_data['minute'] = full_data['timestamp'].apply(lambda x: int(x.split(':')[1]))
full_data = full_data.drop('timestamp', axis=1) # Drop original

# 5. Encode Text to Numbers
text_columns = ['geohash', 'RoadType', 'LargeVehicles', 'Landmarks', 'Weather']
le = LabelEncoder()
for col in text_columns:
    full_data[col] = le.fit_transform(full_data[col].astype(str))

# 6. Split back into Train and Test
train_clean = full_data[full_data['is_train'] == 1].drop('is_train', axis=1)
test_clean = full_data[full_data['is_train'] == 0].drop(['is_train', 'demand'], axis=1)

# 7. SAVE THE FILES LOCALLY IN COLAB
train_clean.to_csv('train_clean.csv', index=False)
test_clean.to_csv('test_clean.csv', index=False)

print("Files created successfully!")

Files created successfully!


In [13]:
from google.colab import files

# Download the cleaned training data
files.download('train_clean.csv')

# Download the cleaned testing data
files.download('test_clean.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [15]:
# Train the Model and Validate

# Separate features (X) and target (y)
X = train_clean.drop(['Index', 'demand'], axis=1)
y = train_clean['demand']

# Split training data into train and local validation sets (80% train, 20% validate)
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialize and train the model
print("Training model... (this might take a minute or two)")
model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

# Make predictions on our local validation set
val_predictions = model.predict(X_val)

# Calculate the Hackathon Score
r2 = r2_score(y_val, val_predictions)
hackathon_score = max(0, 100 * r2)

print(f"Local R2 Score: {r2:.4f}")
print(f"Expected Hackathon Score: {hackathon_score:.2f} / 100")

Training model... (this might take a minute or two)
Local R2 Score: 0.9429
Expected Hackathon Score: 94.29 / 100


In [16]:
# Make final predictions on the test dataset
X_test = test_clean.drop('Index', axis=1)
final_predictions = model.predict(X_test)

# Create the submission dataframe
submission = pd.DataFrame({
    'Index': test_clean['Index'],
    'demand': final_predictions
})

# Save to a CSV file
submission_filename = 'my_first_submission.csv'
submission.to_csv(submission_filename, index=False)

print(f"Success! Saved predictions to {submission_filename}")

Success! Saved predictions to my_first_submission.csv
